# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\rm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\rm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\rm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [2]:
!pip install torch numpy gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.7 MB/s eta 0:00:00


In [3]:
import numpy as np
import torch
import torch.nn as nn
import gensim.downloader as api
from gensim.models import KeyedVectors

def build_embedding_matrix(vocab_limit=100000):
    """
    Google Newsの学習済み単語ベクトルをダウンロードし、
    <PAD>トークンを含む単語埋め込み行列とマッピング辞書を作成する。

    Args:
        vocab_limit (int): 読み込む単語数の上限。Noneを指定すると300万単語すべて読み込む。
                           （※すべて読み込む場合は大容量メモリが必要）
    """
    print("1. Google News学習済み単語ベクトルを取得しています...")
    # return_path=True を指定することで、ダウンロード済みのファイルパスのみを取得
    model_path = api.load('word2vec-google-news-300', return_path=True)

    print(f"2. 単語ベクトルを読み込んでいます (上限: {vocab_limit or 'すべて'} 単語)...")
    # limit引数を使って頻出上位の単語のみを読み込み、メモリを節約
    kv = KeyedVectors.load_word2vec_format(model_path, binary=True, limit=vocab_limit)

    # 3. マッピング辞書の初期化（<PAD> をインデックス 0 として予約）
    # PyTorchでバッチ処理を行う際、短い文と長い文の長さを揃えるため、短い文の末尾にはダミーの単語（パディング）を入れる必要がある
    word2id = {'<PAD>': 0}
    id2word = {0: '<PAD>'}

    # 4. 単語埋め込み行列 E の初期化
    vocab_size = len(kv.index_to_key) + 1  # 読み込んだ単語数 + <PAD>の1枠
    embed_dim = kv.vector_size             # 300次元

    # 全てゼロで初期化。これにより E_{0, :} は自動的にゼロベクトルになる
    E = np.zeros((vocab_size, embed_dim), dtype=np.float32)

    print("3. 単語マッピングと埋め込み行列 E を構築しています...")
    # Gensimモデル内の単語を順に処理
    for i, word in enumerate(kv.index_to_key):
        token_id = i + 1  # 0番は<PAD>なので1から付与

        # 双方向マッピングの構築
        word2id[word] = token_id
        id2word[token_id] = word

        # 行列 E の対応する行にベクトルを代入
        E[token_id] = kv[word]

    # 5. PyTorchのEmbedding層に変換
    # from_pretrained を使い、初期重みとして E を渡す。
    # padding_idx=0 とすることで、ID 0の更新（勾配計算）を防ぐことができる
    embedding_layer = nn.Embedding.from_pretrained(
        embeddings=torch.from_numpy(E),
        padding_idx=0,
        freeze=False  # Trueにすると学習中に重みが更新されない（Fine-tuningしない場合）
    )

    return embedding_layer, word2id, id2word, vocab_size, embed_dim

In [4]:
# 今回はメモリ節約のため、上位10万単語に絞って構築（全300万単語にする場合は None を指定）
LIMIT = 100000

embedding_layer, word2id, id2word, V, d_emb = build_embedding_matrix(vocab_limit=LIMIT)

print("\n=== 構築完了 ===")
print(f"語彙数 |V| (PAD含む): {V}")
print(f"埋め込み次元数 demb: {d_emb}")
print(f"PyTorch Embedding層の重みサイズ: {embedding_layer.weight.shape}")

# <PAD>のベクトルがゼロベクトルであるかの確認
pad_vector = embedding_layer.weight[0].detach().numpy()
is_all_zero = np.all(pad_vector == 0.0)
print(f"インデックス0 (<PAD>) はゼロベクトルか？: {is_all_zero}")

# 特定の単語のIDを確認するテスト
sample_word = "apple"
if sample_word in word2id:
    sample_id = word2id[sample_word]
    print(f"\n'{sample_word}' のトークンID: {sample_id}")
    print(f"ID {sample_id} の単語に復元: {id2word[sample_id]}")

1. Google News学習済み単語ベクトルを取得しています...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
2. 単語ベクトルを読み込んでいます (上限: 100000 単語)...
3. 単語マッピングと埋め込み行列 E を構築しています...

=== 構築完了 ===
語彙数 |V| (PAD含む): 100001
埋め込み次元数 demb: 300
PyTorch Embedding層の重みサイズ: torch.Size([100001, 300])
インデックス0 (<PAD>) はゼロベクトルか？: True

'apple' のトークンID: 13468
ID 13468 の単語に復元: apple


## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

In [1]:
import os
import zipfile
import urllib.request
import pandas as pd
import torch

def download_and_extract_sst2(dest_dir="./SST-2"):
    """
    GLUEベンチマークのSST-2データセットをダウンロードして解凍する関数
    """
    url = "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip"
    zip_name = "SST-2.zip"

    print(f"SST-2 データセットをダウンロード中: {url}")
    urllib.request.urlretrieve(url, zip_name)
    print("解凍中...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("ダウンロードと解凍が完了しました。")


def load_and_preprocess_sst2(file_path, word2id):
    """
    SST-2のTSVファイルを読み込み、指定された条件でトークンID列へ変換する関数
    """
    # TSVファイルの読み込み（quoting=3 でクォーテーション文字を特別扱いせずそのまま読み込む）
    df = pd.read_csv(file_path, sep='\t', quoting=3)

    # 完成データを入れる
    processed_dataset = []
    # スキップした数を数える
    deleted_count = 0

    # Pandasのデータフレームを各行を1つずつ取り出す
    for _, row in df.iterrows():
        text = str(row['sentence'])
        label_val = float(row['label'])

        # テキストをスペースで区切る
        tokens = text.split()

        # 語彙（word2id）に存在する単語のみをIDに変換（語彙外単語は無視して含めない）
        # tokenにテキストをスペースで区切ったtokensを入れて、tokenがword2idに含まれていたら、word2idからtokenをkeyとして取り出す
        input_ids = [word2id[token] for token in tokens if token in word2id]

        # 全てのトークンが語彙外で、空のトークン列になってしまった事例は除外
        if len(input_ids) == 0:
            deleted_count += 1
            continue

        # 指定された辞書オブジェクト形式で保存
        example = {
            'text': text,
            'label': torch.tensor([label_val], dtype=torch.float32),
            'input_ids': torch.tensor(input_ids, dtype=torch.long)
        }
        processed_dataset.append(example)

    print(f"ファイル: {os.path.basename(file_path)}")
    print(f"  - 保持された事例数: {len(processed_dataset)}")
    print(f"  - 空のため削除された事例数: {deleted_count}")

    return processed_dataset

In [8]:
# 1. データのダウンロード
download_and_extract_sst2()

# 2. 訓練セット・開発セットのパス指定
train_tsv_path = "./SST-2/train.tsv"
dev_tsv_path = "./SST-2/dev.tsv"

# 3. 前処理の実行 (70で作成した word2id を使用します)
print("\n--- 訓練セットの前処理を開始 ---")
train_dataset = load_and_preprocess_sst2(train_tsv_path, word2id)

print("\n--- 開発セットの前処理を開始 ---")
dev_dataset = load_and_preprocess_sst2(dev_tsv_path, word2id)

# 4. 変換結果サンプルの確認
print("\n--- 変換結果サンプルの確認 (訓練セットの先頭) ---")
for example in train_dataset:
    if "contains" in example['text']:
        print(example)
        break

SST-2 データセットをダウンロード中: https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
解凍中...
ダウンロードと解凍が完了しました。

--- 訓練セットの前処理を開始 ---
ファイル: train.tsv
  - 保持された事例数: 65837
  - 空のため削除された事例数: 1512

--- 開発セットの前処理を開始 ---
ファイル: dev.tsv
  - 保持された事例数: 872
  - 空のため削除された事例数: 0

--- 変換結果サンプルの確認 (訓練セットの先頭) ---
{'text': 'contains no wit , only labored gags ', 'label': tensor([0.]), 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}


## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [6]:
import torch
import torch.nn as nn

class AverageWordEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx=0, pretrained_embeddings=None):
        """
        単語ベクトルの平均を用いたロジスティック回帰モデル

        Args:
            vocab_size (int): 語彙数 (|V|)
            embed_dim (int): 単語埋め込みの次元数 (demb)
            padding_idx (int): パディングトークンのID (デフォルトは0)
            pretrained_embeddings (torch.Tensor, optional): 事前学習済みの重み行列 E
        """
        super().__init__()
        self.padding_idx = padding_idx

        # 1. 埋め込み層の定義
        if pretrained_embeddings is not None:
            # 事前学習済みの重みを使用する場合
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings,
                freeze=False,
                padding_idx=padding_idx
            )
        else:
            # ランダム初期化から学習する場合
            self.embedding = nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=embed_dim,
                padding_idx=padding_idx
            )

        # 2. ロジスティック回帰用の線形層 (demb次元 -> 1次元)
        # ここでの weight が重みベクトル W、bias が b に相当します
        self.linear = nn.Linear(in_features=embed_dim, out_features=1)

    def forward(self, input_ids):
        """
        順伝播の計算
        Args:
            input_ids: トークンIDのテンソル。形状は (batch_size, max_seq_len)
        Returns:
            probs: ポジティブクラス(1)である確率。形状は (batch_size, 1)
        """
        # 1. トークンIDを単語ベクトルに変換 -> (batch_size, max_seq_len, embed_dim)
        embeds = self.embedding(input_ids)

        # 2. <PAD>トークンを無視するためのマスクを作成
        # IDが padding_idx でない部分を 1.0、padding_idx の部分を 0.0 とする
        # shape: (batch_size, max_seq_len)
        mask = (input_ids != self.padding_idx).float()

        # マスクを次元拡張して埋め込みベクトルに掛け、パディング部分のベクトルを確実に0にする
        # shape: (batch_size, max_seq_len, embed_dim)
        masked_embeds = embeds * mask.unsqueeze(-1)

        # 3. 平均ベクトル v の計算
        # 分子: 単語ベクトルの和 -> (batch_size, embed_dim)
        sum_embeds = masked_embeds.sum(dim=1)

        # 分母: 実際の単語数（パディングを除く） -> (batch_size, 1)
        # ※ 0除算を防ぐため、最小値を 1e-9 にクランプ（制限）します
        valid_lengths = mask.sum(dim=1, keepdim=True).clamp(min=1e-9)

        # 平均ベクトル -> (batch_size, embed_dim)
        avg_embeds = sum_embeds / valid_lengths

        # 4. 線形変換とシグモイド関数による出力
        # (batch_size, 1)
        logits = self.linear(avg_embeds)
        probs = torch.sigmoid(logits)

        return probs

In [9]:
VOCAB_SIZE = 20000
EMBED_DIM = 300
BATCH_SIZE = 2
MAX_SEQ_LEN = 5

# モデルのインスタンス化
model = AverageWordEmbeddingClassifier(
    vocab_size=V,
    embed_dim=d_emb,
    pretrained_embeddings=embedding_layer.weight.clone().detach()
)

# 仮のバッチデータでの動作確認
dummy_input_ids = torch.tensor([
    [3475, 87, 15888, 90, 0],
    [3, 4, 0, 0, 0]
], dtype=torch.long)

model.eval()
with torch.no_grad():
    predictions = model(dummy_input_ids)

print("--- 順伝播のテスト ---")
print(f"入力バッチの形状: {dummy_input_ids.shape}")
print(f"出力確率の形状: {predictions.shape}")
print(predictions)

--- 順伝播のテスト ---
入力バッチの形状: torch.Size([2, 5])
出力確率の形状: torch.Size([2, 1])
tensor([[0.4901],
        [0.4833]])


## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time

# --- データセットクラスの定義 ---
class SST2Dataset(Dataset):
    def __init__(self, data_list, max_len=50, padding_idx=0):
        """
        前処理済みのSST-2データをPyTorchのDatasetとして扱うためのクラス
        """
        self.data_list = data_list
        self.max_len = max_len
        self.padding_idx = padding_idx

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        input_ids = item['input_ids'].clone()
        label = item['label'].clone()

        # パディング処理: max_lenに満たない場合はpadding_idxで埋める
        # 長すぎる場合は切り詰める（今回は簡易的に先頭からmax_lenまで）
        seq_len = input_ids.size(0)
        if seq_len < self.max_len:
            pad_size = self.max_len - seq_len
            pad_tensor = torch.full((pad_size,), self.padding_idx, dtype=torch.long)
            input_ids = torch.cat([input_ids, pad_tensor])
        else:
            input_ids = input_ids[:self.max_len]

        return input_ids, label

# --- 学習関数の定義 ---
def train_model(model, train_dataset, num_epochs=5, batch_size=32, learning_rate=0.01):
    # データローダーの作成
    train_loader = DataLoader(
        SST2Dataset(train_dataset),
        batch_size=batch_size,
        shuffle=True
    )

    # 損失関数: バイナリクロスエントロピー損失 (BCELoss)
    # ※今回はモデルの出力に sigmoid が含まれているため BCELoss を使用します。
    #   もしモデルが出力に logits を返す場合は BCEWithLogitsLoss を使用してください。
    criterion = nn.BCELoss()

    # 1. 埋め込み層のパラメータを固定 (フリーズ)
    for param in model.embedding.parameters():
        param.requires_grad = False

    # 2. オプティマイザ: 線形層のパラメータのみを更新対象とする
    optimizer = optim.Adam(model.linear.parameters(), lr=learning_rate)

    # デバイスの指定 (GPUが使える場合はGPUを使う)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    print(f"--- 学習開始 (デバイス: {device}) ---")
    print(f"学習対象のパラメータ数: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    # 学習ループ
    for epoch in range(num_epochs):
        start_time = time.time()
        model.train() # モデルを訓練モードに設定

        running_loss = 0.0
        correct_preds = 0
        total_samples = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            # 勾配の初期化
            optimizer.zero_grad()

            # 順伝播
            probs = model(inputs)

            # 損失計算
            loss = criterion(probs, labels)

            # 逆伝播
            loss.backward()

            # パラメータの更新
            optimizer.step()

            # モニタリング用の統計情報を記録
            running_loss += loss.item() * inputs.size(0)

            # 予測クラス (0.5以上ならポジティブ)
            preds = (probs >= 0.5).float()
            correct_preds += (preds == labels).sum().item()
            total_samples += inputs.size(0)

        # エポックごとの進捗表示
        epoch_loss = running_loss / total_samples
        epoch_acc = correct_preds / total_samples
        elapsed_time = time.time() - start_time

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Loss: {epoch_loss:.4f} | "
              f"Acc: {epoch_acc:.4f} | "
              f"Time: {elapsed_time:.2f}s")

    print("--- 学習完了 ---")
    return model

In [11]:
# 実際のデータと事前学習済みパラメータを使ってモデルを準備
model_73 = AverageWordEmbeddingClassifier(
    vocab_size=V,
    embed_dim=d_emb,
    pretrained_embeddings=embedding_layer.weight.clone().detach()
)

# 実際の訓練セットを渡して学習を実行
trained_model_73 = train_model(
    model=model_73,
    train_dataset=train_dataset,
    num_epochs=5,
    batch_size=32,
    learning_rate=0.01
)

--- 学習開始 (デバイス: cuda) ---
学習対象のパラメータ数: 301
Epoch 1/5 | Loss: 0.4089 | Acc: 0.8214 | Time: 5.79s
Epoch 2/5 | Loss: 0.3881 | Acc: 0.8305 | Time: 4.25s
Epoch 3/5 | Loss: 0.3868 | Acc: 0.8317 | Time: 4.23s
Epoch 4/5 | Loss: 0.3862 | Acc: 0.8329 | Time: 4.55s
Epoch 5/5 | Loss: 0.3864 | Acc: 0.8318 | Time: 4.07s
--- 学習完了 ---


## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

In [12]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dev_dataset, batch_size=32):
    """
    開発セット上の正解率を計算する関数

    Args:
        model: 学習済みの PyTorch モデル
        dev_dataset: 前処理済みの開発データセットのリスト
        batch_size: バッチサイズ

    Returns:
        accuracy: 正解率 (0.0 ~ 1.0)
    """
    # 評価用データローダーの作成
    dev_loader = DataLoader(
        SST2Dataset(dev_dataset),
        batch_size=batch_size,
        shuffle=False
    )

    # モデルが現在配置されているデバイス(CPU/GPU)を取得
    device = next(model.parameters()).device

    # モデルを評価モードに設定
    model.eval()

    correct_preds = 0
    total_samples = 0

    # 評価時は勾配計算を行わない（メモリ節約・高速化）
    with torch.no_grad():
        # 細かく分けたやつをforループで渡す
        for inputs, labels in dev_loader:
            # バッチデータをモデルと同じ場所に転送
            inputs, labels = inputs.to(device), labels.to(device)

            # 順伝播して予測確率を取得
            probs = model(inputs)

            # 確率が0.5以上ならポジティブ(1.0)、未満ならネガティブ(0.0)と判定
            preds = (probs >= 0.5).float()

            # 正解数をカウント
            correct_preds += (preds == labels).sum().item()
            total_samples += inputs.size(0)

    # 正解率の計算
    accuracy = correct_preds / total_samples

    return accuracy

In [13]:
print("--- 開発セットでの評価を開始 ---")
# 73で学習済みのモデルと、71で作成した実際の開発データを使用
dev_accuracy = evaluate_model(trained_model_73, dev_dataset)

print(f"開発セットの正解率: {dev_accuracy:.4f} ({dev_accuracy * 100:.2f}%)")

--- 開発セットでの評価を開始 ---
開発セットの正解率: 0.7856 (78.56%)


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


In [27]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate(batch):
    """
    可変長の事例リストを受け取り、パディングとソートを行って
    1つのバッチ（辞書形式のテンソル）にまとめる関数。

    Args:
        batch (list): 事例（辞書）のリスト。
                      例: [{'text': ..., 'label': tensor, 'input_ids': tensor}, ...]

    Returns:
        dict: 'input_ids' と 'label' の2つのキーを持つテンソルの辞書。
    """
    # 1. 系列の長さ (input_idsの要素数) を基準に、降順 (長い順) にソートする
    sorted_batch = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)

    # 2. 各要素を抽出してリストにまとめる
    # pad_sequenceに渡すため、input_idsだけのリストを作成
    input_ids_list = [item['input_ids'] for item in sorted_batch]

    # ラベルだけのリストを作成
    labels_list = [item['label'] for item in sorted_batch]

    # 3. input_idsのパディング処理
    # pad_sequence はデフォルトでバッチの最大長に合わせてパディングを行います
    # batch_first=True: 出力の形状を (max_len, batch_size) ではなく (batch_size, max_len) にする
    # padding_value=0: 埋める値として0 (PADトークンID) を指定
    padded_input_ids = pad_sequence(
        input_ids_list,
        batch_first=True,
        padding_value=0
    )

    # 4. ラベルを1つのテンソルに結合する (リストのテンソルを結合)
    # 形状は (batch_size, 1) になります
    labels_tensor = torch.stack(labels_list)

    return {
        'input_ids': padded_input_ids,
        'label': labels_tensor
    }

# ==========================================
# 動作確認
# ==========================================
if __name__ == "__main__":
    # 問題文で提示された4つの事例
    example_batch = [
        {'text': 'hide new secretions from the parental units',
         'label': torch.tensor([0.]),
         'input_ids': torch.tensor([ 5785, 66, 113845, 18, 12, 15095, 1594])},
        {'text': 'contains no wit , only labored gags',
         'label': torch.tensor([0.]),
         'input_ids': torch.tensor([ 3475, 87, 15888, 90, 27695, 42637])},
        {'text': 'that loves its characters and communicates something rather beautiful about human nature',
         'label': torch.tensor([1.]),
         'input_ids': torch.tensor([ 4, 5053, 45, 3305, 31647, 348, 904, 2815, 47, 1276, 1964])},
        {'text': 'remains utterly satisfied to remain the same throughout',
         'label': torch.tensor([0.]),
         'input_ids': torch.tensor([ 987, 14528, 4941, 873, 12, 208, 898])}
    ]

    # collate関数の実行
    batch_out = collate(example_batch)

    # 結果の表示
    print("=== collate関数の出力 ===")
    print("【input_ids】")
    print(batch_out['input_ids'])
    print(f"-> 形状: {batch_out['input_ids'].shape}\n")

    print("【label】")
    print(batch_out['label'])
    print(f"-> 形状: {batch_out['label'].shape}")

=== collate関数の出力 ===
【input_ids】
tensor([[     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,
           1276,   1964],
        [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,
              0,      0],
        [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,
              0,      0],
        [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,
              0,      0]])
-> 形状: torch.Size([4, 11])

【label】
tensor([[1.],
        [0.],
        [0.],
        [0.]])
-> 形状: torch.Size([4, 1])


## 76. ミニバッチ学習

問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import time

class SimpleSST2Dataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

# ==========================================
# 2. 学習および評価関数
# ==========================================

def train_and_evaluate(model, train_data, dev_data, num_epochs=5, batch_size=32, lr=0.01):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # DataLoaderの作成 (ここで collate_fn=collate を指定)
    train_loader = DataLoader(
        SimpleSST2Dataset(train_data),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate
    )
    dev_loader = DataLoader(
        SimpleSST2Dataset(dev_data),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate
    )

    criterion = nn.BCELoss()
    # 埋め込み層を固定し、線形層のみを学習
    for param in model.embedding.parameters():
        param.requires_grad = False
    optimizer = optim.Adam(model.linear.parameters(), lr=lr)

    print(f"--- 学習開始 (デバイス: {device}) ---")

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        start_time = time.time()

        # --- 訓練フェーズ ---
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            probs = model(inputs)
            loss = criterion(probs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = (probs >= 0.5).float()
            correct_train += (preds == labels).sum().item()
            total_train += inputs.size(0)

        train_loss = running_loss / total_train
        train_acc = correct_train / total_train

        # --- 評価フェーズ (開発セット) ---
        model.eval()
        correct_dev = 0
        total_dev = 0

        with torch.no_grad():
            for batch in dev_loader:
                inputs = batch['input_ids'].to(device)
                labels = batch['label'].to(device)

                probs = model(inputs)
                preds = (probs >= 0.5).float()
                correct_dev += (preds == labels).sum().item()
                total_dev += inputs.size(0)

        dev_acc = correct_dev / total_dev
        elapsed_time = time.time() - start_time

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Dev Acc: {dev_acc:.4f} | "
              f"Time: {elapsed_time:.2f}s")

    return model, dev_acc

# 引数から pretrained_embeddings を削除
model_76 = AverageWordEmbeddingClassifier(
    vocab_size=V,
    embed_dim=d_emb
)

# 構築後に事前学習済みの重みを直接上書き（コピー）する
model_76.embedding.weight.data.copy_(embedding_layer.weight.clone().detach())

trained_model_76, final_dev_acc_76 = train_and_evaluate(
    model=model_76,
    train_data=train_dataset,
    dev_data=dev_dataset,
    num_epochs=5,
    batch_size=32,
    lr=0.01
)

--- 学習開始 (デバイス: cuda) ---
Epoch 1/5 | Train Loss: 0.6954 | Train Acc: 0.5190 | Dev Acc: 0.5400 | Time: 0.20s
Epoch 2/5 | Train Loss: 0.6811 | Train Acc: 0.5640 | Dev Acc: 0.5400 | Time: 0.07s
Epoch 3/5 | Train Loss: 0.6723 | Train Acc: 0.5860 | Dev Acc: 0.5350 | Time: 0.06s
Epoch 4/5 | Train Loss: 0.6641 | Train Acc: 0.6350 | Dev Acc: 0.5300 | Time: 0.06s
Epoch 5/5 | Train Loss: 0.6574 | Train Acc: 0.6490 | Dev Acc: 0.5300 | Time: 0.06s

開発セットにおける正解率: 0.5300


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import time


# ==========================================
# 2. 【GPU対応】学習および評価関数
# ==========================================
def train_and_evaluate_gpu(model, train_data, dev_data, num_epochs=5, batch_size=32, lr=0.01):
    # ---------------------------------------------------------
    # デバイスの判定とモデルの転送
    # ---------------------------------------------------------
    if torch.cuda.is_available():
        device = torch.device("cuda")       # NVIDIA GPU
    elif torch.backends.mps.is_available():
        device = torch.device("mps")        # Apple Silicon (M1/M2/M3) Mac
    else:
        device = torch.device("cpu")        # GPUがない場合はCPU

    print(f"--- 実行デバイス: {device} ---")

    # モデルの全パラメータをデバイス(GPU)メモリに転送
    model.to(device)

    # 訓練用　データのリストを、データセット形式にする
    train_loader = DataLoader(
        SimpleSST2Dataset(train_data), batch_size=batch_size, shuffle=True, collate_fn=collate
    )
    # 評価用
    dev_loader = DataLoader(
        SimpleSST2Dataset(dev_data), batch_size=batch_size, shuffle=False, collate_fn=collate
    )

    # 損失関数を定義
    criterion = nn.BCELoss()

    # 埋め込み層を固定
    for param in model.embedding.parameters():
        param.requires_grad = False

    # オプティマイザの初期化 (model.to(device) の後に初期化することが重要です)
    optimizer = optim.Adam(model.linear.parameters(), lr=lr)

    print("--- 学習開始 ---")
    # エポックごとにループ
    for epoch in range(num_epochs):
        # [訓練フェーズ]
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        start_time = time.time()

        for batch in train_loader:
            # ---------------------------------------------------------
            # バッチデータをデバイス(GPU)へ転送
            # ---------------------------------------------------------
            inputs = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            # 前回計算した勾配（傾き）をリセット
            optimizer.zero_grad()
            # 予測
            probs = model(inputs)
            # 誤差を逆伝播させ、勾配を計算
            loss = criterion(probs, labels)
            loss.backward()

            # モデルの重みを更新
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = (probs >= 0.5).float()
            correct_train += (preds == labels).sum().item()
            total_train += inputs.size(0)

        train_loss = running_loss / total_train
        train_acc = correct_train / total_train

        # [評価フェーズ (開発セット)]
        model.eval()
        correct_dev = 0
        total_dev = 0

        # 評価時は重みを更新しないため、勾配の計算を停止
        with torch.no_grad():
            for batch in dev_loader:
                # ---------------------------------------------------------
                # 評価時もバッチデータをデバイス(GPU)へ転送
                # ---------------------------------------------------------
                inputs = batch['input_ids'].to(device)
                labels = batch['label'].to(device)

                # 予測
                probs = model(inputs)
                # モデルが出力した確率を 0.5 を境に判定
                # 0.5 以上なら True, 0.5 未満なら False
                # True → 1.0, False → 0.0に変換
                preds = (probs >= 0.5).float()

                # 今回のバッチで、いくつ正解したかをカウント
                correct_dev += (preds == labels).sum().item()

                # 全部で何個のデータを判定したかをカウント
                total_dev += inputs.size(0)

        # 評価データにおける正解率
        dev_acc = correct_dev / total_dev
        # このエポック1周にかかった経過時間
        elapsed_time = time.time() - start_time

        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Dev Acc: {dev_acc:.4f} | "
              f"Time: {elapsed_time:.2f}s")

    return model, dev_acc

In [35]:
# 訓練データ（1000個）
train_dataset = generate_mock_data(1000)
# 評価データ（200個）
dev_dataset = generate_mock_data(200)

model = AverageWordEmbeddingClassifier(vocab_size=10000, embed_dim=300)

trained_model, final_dev_acc = train_and_evaluate_gpu(
    model=model,
    train_data=train_dataset,
    dev_data=dev_dataset,
    num_epochs=5,      # 5回繰り返して学習
    batch_size=32,     # 32個ずつまとめて処理
    lr=0.005           # 学習率
)

print("\n=== 最終結果 ===")
print(f"開発セットにおける正解率: {final_dev_acc:.4f} ({final_dev_acc * 100:.2f}%)")

--- 実行デバイス: cuda ---
--- 学習開始 ---
Epoch 01/5 | Train Loss: 0.7000 | Train Acc: 0.5110 | Dev Acc: 0.4900 | Time: 0.07s
Epoch 02/5 | Train Loss: 0.6525 | Train Acc: 0.6390 | Dev Acc: 0.5100 | Time: 0.06s
Epoch 03/5 | Train Loss: 0.6255 | Train Acc: 0.6670 | Dev Acc: 0.4900 | Time: 0.06s
Epoch 04/5 | Train Loss: 0.6065 | Train Acc: 0.6970 | Dev Acc: 0.4900 | Time: 0.06s
Epoch 05/5 | Train Loss: 0.5920 | Train Acc: 0.7040 | Dev Acc: 0.4700 | Time: 0.06s

=== 最終結果 ===
開発セットにおける正解率: 0.4700 (47.00%)


## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import time

# ==========================================
# 1. コンポーネント定義 (前回と同様)
# ==========================================
def collate(batch):
    sorted_batch = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)
    input_ids_list = [item['input_ids'] for item in sorted_batch]
    labels_list = [item['label'] for item in sorted_batch]
    padded_input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=0)
    labels_tensor = torch.stack(labels_list)
    return {'input_ids': padded_input_ids, 'label': labels_tensor}

class AverageWordEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx=0):
        super().__init__()
        self.padding_idx = padding_idx
        # ※実際の実装では nn.Embedding.from_pretrained(pretrained_weights, freeze=False, padding_idx=0) を用います
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.linear = nn.Linear(embed_dim, 1)

    def forward(self, input_ids):
        embeds = self.embedding(input_ids)
        mask = (input_ids != self.padding_idx).float()
        masked_embeds = embeds * mask.unsqueeze(-1)
        sum_embeds = masked_embeds.sum(dim=1)
        valid_lengths = mask.sum(dim=1, keepdim=True).clamp(min=1e-9)
        avg_embeds = sum_embeds / valid_lengths
        return torch.sigmoid(self.linear(avg_embeds))

class SimpleSST2Dataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

# ==========================================
# 2. 【ファインチューニング対応】学習および評価関数
# ==========================================
def train_and_evaluate_finetune(model, train_data, dev_data, num_epochs=5, batch_size=32, lr=0.001):
    # デバイスの設定
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    print(f"--- 実行デバイス: {device} ---")
    model.to(device)

    train_loader = DataLoader(
        SimpleSST2Dataset(train_data), batch_size=batch_size, shuffle=True, collate_fn=collate
    )
    dev_loader = DataLoader(
        SimpleSST2Dataset(dev_data), batch_size=batch_size, shuffle=False, collate_fn=collate
    )

    criterion = nn.BCELoss()

    # ---------------------------------------------------------
    # 【変更点】
    # 1. embeddingの requires_grad = False (フリーズ) の処理を削除しました。
    #    (PyTorchではデフォルトで requires_grad = True となっています)
    #
    # 2. オプティマイザの更新対象を model.linear.parameters() から
    #    model.parameters() (モデル全体のパラメータ) に変更しました。
    # ---------------------------------------------------------
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print(f"学習対象のパラメータ数: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
    print("--- 学習開始 (Fine-tuning) ---")

    for epoch in range(num_epochs):
        # [訓練フェーズ]
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        start_time = time.time()

        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            probs = model(inputs)
            loss = criterion(probs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            preds = (probs >= 0.5).float()
            correct_train += (preds == labels).sum().item()
            total_train += inputs.size(0)

        train_loss = running_loss / total_train
        train_acc = correct_train / total_train

        # [評価フェーズ (開発セット)]
        model.eval()
        correct_dev = 0
        total_dev = 0

        with torch.no_grad():
            for batch in dev_loader:
                inputs = batch['input_ids'].to(device)
                labels = batch['label'].to(device)

                probs = model(inputs)
                preds = (probs >= 0.5).float()
                correct_dev += (preds == labels).sum().item()
                total_dev += inputs.size(0)

        dev_acc = correct_dev / total_dev
        elapsed_time = time.time() - start_time

        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Dev Acc: {dev_acc:.4f} | "
              f"Time: {elapsed_time:.2f}s")

    return model, dev_acc

# ==========================================
# 3. 実行ブロック (モックデータでのテスト)
# ==========================================
if __name__ == "__main__":
    import random

    def generate_mock_data(num_samples):
        data = []
        for _ in range(num_samples):
            seq_len = random.randint(3, 15)
            input_ids = torch.randint(1, 10000, (seq_len,))
            label = torch.tensor([random.choice([0.0, 1.0])])
            data.append({'input_ids': input_ids, 'label': label})
        return data

    train_dataset = generate_mock_data(1000)
    dev_dataset = generate_mock_data(200)

    model = AverageWordEmbeddingClassifier(vocab_size=10000, embed_dim=300)

    # ファインチューニング時は、事前学習済みの表現を壊さないように
    # 学習率(lr)を少し小さめ（例: 0.01 -> 0.001など）に設定するのが一般的です
    trained_model, final_dev_acc = train_and_evaluate_finetune(
        model=model,
        train_data=train_dataset,
        dev_data=dev_dataset,
        num_epochs=5,
        batch_size=32,
        lr=0.001
    )

    print("\n=== 最終結果 ===")
    print(f"開発セットにおける正解率: {final_dev_acc:.4f} ({final_dev_acc * 100:.2f}%)")

--- 実行デバイス: cuda ---
学習対象のパラメータ数: 3000301
--- 学習開始 (Fine-tuning) ---
Epoch 01/5 | Train Loss: 0.7005 | Train Acc: 0.4920 | Dev Acc: 0.5800 | Time: 0.12s
Epoch 02/5 | Train Loss: 0.6721 | Train Acc: 0.6110 | Dev Acc: 0.5800 | Time: 0.09s
Epoch 03/5 | Train Loss: 0.6457 | Train Acc: 0.6910 | Dev Acc: 0.5950 | Time: 0.10s
Epoch 04/5 | Train Loss: 0.6133 | Train Acc: 0.7870 | Dev Acc: 0.5850 | Time: 0.09s
Epoch 05/5 | Train Loss: 0.5728 | Train Acc: 0.8500 | Dev Acc: 0.5900 | Time: 0.10s

=== 最終結果 ===
開発セットにおける正解率: 0.5900 (59.00%)


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import time

# ==========================================
# 1. データセットとパディング処理 (前ステップと同一)
# ==========================================
def collate(batch):
    sorted_batch = sorted(batch, key=lambda x: x['input_ids'].size(0), reverse=True)
    input_ids_list = [item['input_ids'] for item in sorted_batch]
    labels_list = [item['label'] for item in sorted_batch]
    padded_input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=0)
    labels_tensor = torch.stack(labels_list)
    return {'input_ids': padded_input_ids, 'label': labels_tensor}

class SimpleSST2Dataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx]

# ==========================================
# 2. 【新規】BiLSTMモデルの定義
# ==========================================
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, padding_idx=0, dropout=0.5):
        """
        双方向LSTMを用いたテキスト分類モデル
        """
        super().__init__()
        self.padding_idx = padding_idx

        # 1. 埋め込み層
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

        # 2. 双方向LSTM層
        # bidirectional=True にすることで、出力の次元数は hidden_dim * 2 になります
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # 3. 全結合層 (分類器)
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        # 1. 単語埋め込み -> (batch_size, seq_len, embed_dim)
        embeds = self.embedding(input_ids)

        # 2. LSTMの適用 -> (batch_size, seq_len, hidden_dim * 2)
        lstm_out, _ = self.lstm(embeds)

        # 3. パディング部分のマスク作成
        # padding_idx の場所は 0、それ以外は 1 となるマスク
        mask = (input_ids != self.padding_idx).unsqueeze(-1).float()

        # パディング部分の出力を負の無限大に近い値(-1e9)で埋める
        # これにより、次の Max Pooling でパディング部分が最大値として選ばれるのを防ぎます
        masked_out = lstm_out.masked_fill(mask == 0, -1e9)

        # 4. Max Pooling (時間軸 dim=1 に沿って最大値を取得) -> (batch_size, hidden_dim * 2)
        max_pool_out, _ = torch.max(masked_out, dim=1)

        # 5. 全結合層による分類
        x = self.dropout(max_pool_out)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        logits = self.fc2(x)

        return torch.sigmoid(logits)

# ==========================================
# 3. GPU対応の学習・評価関数 (前ステップと同一)
# ==========================================
def train_and_evaluate_model(model, train_data, dev_data, num_epochs=5, batch_size=32, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
    print(f"--- 実行デバイス: {device} ---")
    model.to(device)

    train_loader = DataLoader(SimpleSST2Dataset(train_data), batch_size=batch_size, shuffle=True, collate_fn=collate)
    dev_loader = DataLoader(SimpleSST2Dataset(dev_data), batch_size=batch_size, shuffle=False, collate_fn=collate)

    criterion = nn.BCELoss()
    # モデル全体（単語埋め込みも含む）をファインチューニング対象とする
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print("--- 学習開始 ---")
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        start_time = time.time()

        for batch in train_loader:
            inputs, labels = batch['input_ids'].to(device), batch['label'].to(device)
            optimizer.zero_grad()
            probs = model(inputs)
            loss = criterion(probs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            correct_train += ((probs >= 0.5).float() == labels).sum().item()
            total_train += inputs.size(0)

        train_loss = running_loss / total_train
        train_acc = correct_train / total_train

        model.eval()
        correct_dev, total_dev = 0, 0
        with torch.no_grad():
            for batch in dev_loader:
                inputs, labels = batch['input_ids'].to(device), batch['label'].to(device)
                probs = model(inputs)
                correct_dev += ((probs >= 0.5).float() == labels).sum().item()
                total_dev += inputs.size(0)

        dev_acc = correct_dev / total_dev
        elapsed_time = time.time() - start_time
        print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Dev Acc: {dev_acc:.4f} | Time: {elapsed_time:.2f}s")

    return model, dev_acc

# ==========================================
# 4. 実行ブロック (モックデータでのテスト)
# ==========================================
if __name__ == "__main__":
    import random
    def generate_mock_data(num_samples):
        data = []
        for _ in range(num_samples):
            seq_len = random.randint(5, 20)
            input_ids = torch.randint(1, 10000, (seq_len,))
            label = torch.tensor([random.choice([0.0, 1.0])])
            data.append({'input_ids': input_ids, 'label': label})
        return data

    train_dataset = generate_mock_data(1000)
    dev_dataset = generate_mock_data(200)

    # モデルのインスタンス化 (隠れ層の次元数 hidden_dim を指定)
    model = BiLSTMClassifier(vocab_size=10000, embed_dim=300, hidden_dim=128)

    # 学習の実行
    trained_model, final_dev_acc = train_and_evaluate_model(
        model=model, train_data=train_dataset, dev_data=dev_dataset, num_epochs=5, batch_size=32, lr=0.001
    )

    print("\n=== 最終結果 (BiLSTM) ===")
    print(f"開発セットにおける正解率: {final_dev_acc:.4f} ({final_dev_acc * 100:.2f}%)")

--- 実行デバイス: cuda ---
--- 学習開始 ---
Epoch 01/5 | Train Loss: 0.6981 | Train Acc: 0.5050 | Dev Acc: 0.5400 | Time: 0.53s
Epoch 02/5 | Train Loss: 0.6797 | Train Acc: 0.5750 | Dev Acc: 0.5450 | Time: 0.15s
Epoch 03/5 | Train Loss: 0.6348 | Train Acc: 0.7120 | Dev Acc: 0.5200 | Time: 0.15s
Epoch 04/5 | Train Loss: 0.4823 | Train Acc: 0.8090 | Dev Acc: 0.4800 | Time: 0.15s
Epoch 05/5 | Train Loss: 0.1564 | Train Acc: 0.9670 | Dev Acc: 0.5000 | Time: 0.16s

=== 最終結果 (BiLSTM) ===
開発セットにおける正解率: 0.5000 (50.00%)
